# obspec_utils smoke test

`obspec-utils` provides utilities on top of the `obspec` protocols: glob
matching, a URL→store registry, file-like readers, and caching/tracing
wrappers. All examples run offline against `obstore.store.MemoryStore`.

## Example 1: glob pattern matching over an object store

In [ ]:
# Example 1: glob pattern matching over any obspec List store
# Source: adapted from the glob docstring in obspec_utils/glob.py (packaged wheel)
from obstore.store import MemoryStore

from obspec_utils import glob, glob_objects

store = MemoryStore()
for path in [
    "data/2024/jan.nc", "data/2024/feb.nc", "data/2023/dec.nc",
    "data/readme.txt", "logs/run1.log",
]:
    store.put(path, b"x" * 8)

assert sorted(glob(store, "data/2024/*.nc")) == ["data/2024/feb.nc", "data/2024/jan.nc"]
assert len(list(glob(store, "data/**/*.nc"))) == 3
assert list(glob(store, "logs/run?.log")) == ["logs/run1.log"]

metas = list(glob_objects(store, "data/**/*.nc"))
assert all(meta["size"] == 8 for meta in metas)
print("glob matched:", sorted(glob(store, "data/**/*.nc")))


## Example 2: `ObjectStoreRegistry` URL resolution

In [ ]:
# Example 2: ObjectStoreRegistry maps URLs to stores
# Source: adapted from the ObjectStoreRegistry docstring in obspec_utils/registry.py (packaged wheel)
from obstore.store import MemoryStore

from obspec_utils.registry import ObjectStoreRegistry

bucket_a, bucket_b = MemoryStore(), MemoryStore()
registry = ObjectStoreRegistry({
    "s3://bucket-a": bucket_a,
    "s3://bucket-b/prefix": bucket_b,
})

store, path = registry.resolve("s3://bucket-a/data/file.nc")
assert store is bucket_a and path == "data/file.nc"

store, path = registry.resolve("s3://bucket-b/prefix/other.nc")
assert store is bucket_b and path == "prefix/other.nc"  # path is relative to scheme+netloc

# register() adds mappings after construction
local = MemoryStore()
registry.register("file://", local)
store, _ = registry.resolve("file:///tmp/x.bin")
assert store is local
print("resolved:", registry.resolve("s3://bucket-a/data/file.nc"))


## Example 3: file-like readers

In [ ]:
from obstore.store import MemoryStore

from obspec_utils.readers import BufferedStoreReader, EagerStoreReader

store = MemoryStore()
payload = bytes(range(256)) * 40  # 10,240 bytes
store.put("blobs/data.bin", payload)

reader = BufferedStoreReader(store, "blobs/data.bin", buffer_size=1024)
assert reader.read(16) == payload[:16]
assert reader.tell() == 16
reader.seek(5000)
assert reader.read(100) == payload[5000:5100]
reader.seek(-256, 2)  # seek relative to end
assert reader.read() == payload[-256:]

eager = EagerStoreReader(store, "blobs/data.bin")
assert eager.read() == payload
print(f"BufferedStoreReader + EagerStoreReader OK over {len(payload)} bytes")

## Example 4: caching + tracing wrappers

In [ ]:
# Example 4: TracingReadableStore observes requests; CachingReadableStore dedupes them
# Source: adapted from the TracingReadableStore docstring in obspec_utils/wrappers/_tracing.py (packaged wheel)
from obstore.store import MemoryStore

from obspec_utils.wrappers import CachingReadableStore, RequestTrace, TracingReadableStore

store = MemoryStore()
store.put("data/chunk.bin", b"A" * 4096)

trace = RequestTrace()
traced = TracingReadableStore(store, trace)
cached = CachingReadableStore(traced)

for _ in range(3):
    data = cached.get_range("data/chunk.bin", start=0, end=1024)
assert bytes(data) == b"A" * 1024

hits = [r for r in trace.requests if r.path == "data/chunk.bin"]
print(f"3 reads -> {len(hits)} backend request(s): {[r.method for r in hits]}")
assert len(hits) < 3  # the cache fills once; later reads never reach the backend


## Example 5: async glob (top-level await)

In [ ]:
from obstore.store import MemoryStore

from obspec_utils import glob_async, glob_objects_async

store = MemoryStore()
for path in ["a/x.nc", "a/y.nc", "b/z.nc", "a/notes.md"]:
    await store.put_async(path, b"data")

names = [p async for p in glob_async(store, "a/*.nc")]
assert sorted(names) == ["a/x.nc", "a/y.nc"]

metas = [m async for m in glob_objects_async(store, "**/*.nc")]
assert len(metas) == 3 and all(m["size"] == 4 for m in metas)
print("async glob:", sorted(names), "| objects:", len(metas))